In [1]:
import math
import torch
import random
import numpy as np
from torch import nn, optim

from datetime import datetime
from dataclasses import dataclass, asdict
from tokenizer import TinyStoriesTokenizer
from self_attention import MultiHeadSelfAttention

## Transformer block
To obtain a complete transformer block, we add a __position-wise feed-forward layer__ after the self-attention computation. The vectors are first projected to be 4 times wider, a non-linear ``relu'' function is applied, and the vectors are projected down to their original size.

Before and after the self-attention computation, we add a __layer norm__ which normalizes each vector to have mean 0 and variance 1. This contributes to "numerical stability" in the forward pass, i.e. that the numbers within the vectors stay within the bounds representable by a computer. 

We also add __residual connections__ to each layer. In general, this means that we compute $x+f(x)$ rather than just $f(x)$ (the "$x+$" is sometimes called the "identity shortcut"). The gradient of $x+f(x)$ is $1+f'(x)$, which is close to 1 if $f'(x)$ is tiny. Without the identity shortcut, the overall gradient in the backward pass could be come extremely small, in particular in a deep network where many layers are stacked. The residual connection thus acts as a prevention for the "vanishing gradient" problem.



In [2]:
class PositionwiseFFN(nn.Module):
    """
    The position-wise FFN that follows after the self-attention computation.
    Vectors are projected to 4x the dimensionality and then projected down
    again after relu application.
    """

    def __init__(self, vector_dim, dropout_prob) :
        super().__init__()
        self.fc1 = nn.Linear(vector_dim, 4*vector_dim, bias=True)
        self.fc2 = nn.Linear(4*vector_dim, vector_dim, bias=True)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

class Block(nn.Module):
    """
    Transformer encoder block.

    This version differs from the original version in  [Vaswani et al. NeurIPS 2017],
    and applies the LayerNorm before the self-attention, and before the FFN, as this
    has proved to be beneficial (see [Nguyen and Salazar 2019]).
    """

    def __init__(self, vector_dim, n_heads, block_size, dropout_prob):
        super().__init__()
        att_dim = vector_dim // n_heads
        self.attn = MultiHeadSelfAttention(vector_dim, n_heads, block_size, is_causal=False) #MODIFY IMPO
        self.ffn = PositionwiseFFN(vector_dim, dropout_prob)
        self.dropout = nn.Dropout(dropout_prob)
        self.ln1 = nn.LayerNorm(vector_dim)
        self.ln2 = nn.LayerNorm(vector_dim)

    def forward(self, x):
        x1 = self.ln1(x) # pre layer norm
        x2 = x + self.dropout(self.attn(x1)) # attention + residual
        x3 = self.ln2(x2)
        x4 = x2 + self.dropout(self.ffn(x3)) # ffn + residual
        return x4

## Fill in the blanks

In [3]:
# ============= Hyper-parameters for training ============== #

@dataclass
class Config :
    vocab_size: int = 5000 + 1  # This number should agree with the tokenizer
    mask_token_id: int = 5000  # new token forn unmasking
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 4
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesLM(nn.Module):

    def __init__(self, config):
        super(TinyStoriesLM, self).__init__()
        self.config = config
        self.embed =  nn.Embedding(config.vocab_size, config.vector_dim)
        self.positional = nn.Parameter(torch.randn(1, config.block_size, config.vector_dim)) # learns during trianing, beginning random
        modules = [Block(config.vector_dim,\
                         config.number_of_attention_heads,\
                         config.block_size,\
                         config.dropout_prob) for _ in range(config.number_of_transformer_blocks)]
        self.transformers = nn.ModuleList(modules)
        self.final = nn.Linear(config.vector_dim, config.vocab_size)

    def forward(self, x):
        # x size (B, S)
        # each element is a token-word -> transf it to vector
        B, S = x.shape

        # Embedding (B, S, vector_dim)
        token_embeddings = self.embed(x) # each num.token is now a 256 len vector

        # Position Embedding - size ( 1, blocksize, vector_dim) -> S increasing as model generates more words
        pos_embeddings = self.positional[:, :S, :] # (1, S, vector_dim)

        x = token_embeddings + pos_embeddings # (B, S, vector_dim)

        # TRANSFORMERS BLOCK
        for block in self.transformers:
            x = block(x)
        # output - same len but with more context

        # Project to 5_001 words. For each pos - how probable for next word to be ... from 0-4999
        logits = self.final(x) # (B, S, vector_dim) to (B, S, vocab_size)

        # YOUR CODE HERE

        return logits

    @classmethod
    def load(cls, checkpoint_path, device='cpu'):
        """
        Loads a model from a checkpoint file.
        Automatically reconstructs the config and model architecture.
        """
        checkpoint = torch.load(checkpoint_path, map_location=device)
        config = Config(**checkpoint['config'])
        model = cls(config)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']}, iteration {checkpoint['iteration']})")
        return model



In [4]:
from torch.utils.data import Sampler, SubsetRandomSampler
from torch.utils.data import Dataset, DataLoader
from tokenizer import TinyStoriesTokenizer

class MaskedTinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size, vocab_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size
        self.mask_token_id = vocab_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):

        """
        For each input sentence, randomly select 15% of the tokens.
        We call these “selected tokens”. Of the selected tokens, and replace 80% with “[MASK]”, 10% with a random word, and leave 10% unchanged.
        Train the model to predict the original identity of all selected tokens.
        Note that your loss function should only be calculated over the selected tokens
        """
        # Pull a chunk of length block_size 
        chunk = self.data[idx : idx + self.block_size]
        chunk = torch.from_numpy(chunk.astype(np.int64))
        original = chunk.clone()     # original
        masked_input = chunk.clone() # modified

        # randomly select 15%
        probs = torch.full(original.shape, 0.15)
        selected_idx = torch.bernoulli(probs).bool()

        # 15% si what we want to predict
        # From 15%:

        for idx in selected_idx.nonzero(as_tuple=True)[0]: # give me the idx of the rows where true in selected_idx
          r = random.random()
          if r < 0.80:
              masked_input[idx] = self.mask_token_id # 80% time we replace with mask [MASK]
          elif r < 0.90:
              masked_input[idx] = random.randint(0, self.mask_token_id-1)  # 10% replace with random word, exlcuidng mask
          # else: 10% remains the same
        # masked input = The [MASK] sat on the [RANDOM: airplane]

        return masked_input, original, selected_idx


# This class facilitates re-starting training without repeating any past training examples
class ResumableRandomSampler(Sampler):
    def __init__(self, data_source, start_index=0, seed=42):
        self.data_source = data_source
        self.start_index = start_index
        self.seed = seed
        self.generator = torch.Generator()
        self.generator.manual_seed(self.seed)

        # 1. Generate the full shuffled list of indices once
        self.indices = torch.randperm(len(self.data_source), generator=self.generator).tolist()

    def __iter__(self):
        # 2. Slice the indices to only include those from start_index onwards
        return iter(self.indices[self.start_index:])

    def __len__(self):
        return len(self.indices) - self.start_index

In [6]:
import os

# ======================= Training ======================= #

# Remove the checkpoint file if you want to train from scratch
checkpoint_path = 'last_checkpoint_unmasking.pt'
RESUME_TRAINING = os.path.exists(checkpoint_path)

seed = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print( "Running on", device )

tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')

# IMPO Add [MASK] to tokens
mask_token = "[MASK]"
if mask_token not in tokenizer.vocab:
    tokenizer.vocab.append(mask_token)
    tokenizer.vocab_size += 1
mask_id = len(tokenizer.vocab) - 1 

if RESUME_TRAINING:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']+1}, iteration {checkpoint['iteration']})")
    config = Config(**checkpoint['config'])
    lm = TinyStoriesLM(config)
    # Sanity check
    if tokenizer.vocab_size != lm.config.vocab_size:
        print("WARNING: The tokenizer's and the model's vocab_size are different!")
    lm.load_state_dict(checkpoint['model_state_dict'])
    lm.to(device)
else :
    config = Config()
    config.vocab_size = tokenizer.vocab_size  # 5001
    config.mask_token_id = tokenizer.vocab_size - 1  # 5000
    print('len tokenizer: ', len(tokenizer.vocab))
    lm = TinyStoriesLM(config).to(device)

lm_optimizer = torch.optim.AdamW(lm.parameters(), lr=lm.config.learning_rate, weight_decay=config.weight_decay)

if RESUME_TRAINING:
    lm_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

training_dataset = MaskedTinyStoriesDataset('/datasets/dd2417/train.bin', config.block_size, config.mask_token_id)

# Development set
dev_dataset = MaskedTinyStoriesDataset('/datasets/dd2417/dev.bin', config.block_size, config.mask_token_id)
# Create indices that jump by the block size. This ensures we only see each
# chunk of text once when computing validation loss.
dev_indices = list(range(0, len(dev_dataset), config.block_size))
dev_subset = torch.utils.data.Subset(dev_dataset, dev_indices)
dev_loader = DataLoader(dev_subset, config.batch_size, pin_memory=True)

print( "There are", len(training_dataset), "datapoints and", tokenizer.vocab_size, "token classes in the dataset" )

criterion = nn.CrossEntropyLoss()

lm.train()
running_loss = 0
print( datetime.now().strftime("%X"), "Training starts" )
# Can we pick up where we left off?
start_epoch = checkpoint['epoch'] if RESUME_TRAINING else 0
start_iter = checkpoint['iteration'] if RESUME_TRAINING else 0

# CHANGE
for epoch in range(start_epoch, config.no_of_epochs) :
    iteration = start_iter if epoch == start_epoch else 0
    # Random sampling of datapoints. Note that we use a new seed for each epoch
    # -- this will guarantee that we can resume the training anywhere without
    # repeating any datapoints within an epoch.
    sampler = ResumableRandomSampler(training_dataset,
                                     start_index=iteration * config.batch_size,
                                     seed=seed+epoch)
    training_loader = DataLoader(training_dataset,
                                 batch_size=config.batch_size,
                                 sampler=sampler,
                                 num_workers=4)
    for masked_input, original, selected in training_loader:
        masked_input, original, selected = masked_input.to(device), original.to(device), selected.to(device)
        lm_optimizer.zero_grad() # erase previous error

        # Prediction forward
        logits = lm(masked_input) 
        # masked input (15% - 80-10-10)-> what model sees | original -> what to predict | selected -> boolean, where calculate loss (random 15%)
        loss = criterion(logits[selected],  original[selected] )
        loss.backward() # gradients
        lm_optimizer.step() # update weights


        running_loss += loss.detach().item()
        iteration += 1
        if iteration%100 == 0:
            # Compute validation loss every 100 iterations
            # CHANGE
            lm.eval()
            with torch.no_grad():
                total_loss = 0
                for vx, vy, v_selected in dev_loader:
                  vx, vy, v_selected = vx.to(device), vy.to(device), v_selected.to(device)
                  logits = lm(vx)
                  val_loss = criterion(logits[v_selected], vy[v_selected])
                  total_loss += val_loss.item()
                print(f'{datetime.now().strftime("%X")} Epoch {epoch+1}, iteration {iteration}: loss={running_loss/100:.4f}, dev loss={total_loss/len(dev_loader):.4f}')
            lm.train()
            running_loss = 0
        if iteration%10000 == 0:
            checkpoint = {
                'model_state_dict': lm.state_dict(),
                'optimizer_state_dict': lm_optimizer.state_dict(),
                'epoch': epoch,
                'iteration': iteration,
                'config': config.__dict__, # Useful so you know the architecture later
            }

            # Overwrites the same file every time to save space
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Show progress
            lm.eval()
            with torch.no_grad():
              sample_input, sample_original, sample_selected = next(iter(dev_loader))
              sample_input = sample_input.to(device)
              sample_original = sample_original.to(device)
              sample_selected = sample_selected.to(device)
              
              logits = lm(sample_input)
              predictions = logits.argmax(dim=-1)  # token max probability
              
              # Get the indx for selected -> jsut print the first 5 tokens first example from batch
              selected_positions = sample_selected[0].nonzero(as_tuple=True)[0][:5]
              print("-" * 30)
              for pos in selected_positions:
                  predicted = tokenizer.vocab[predictions[0][pos].item()]
                  original  = tokenizer.vocab[sample_original[0][pos].item()]
                  masked    = sample_input[0][pos].item()
                  print(f"original: {original:15} | prediction: {predicted:15} | is [MASK]?: {masked == config.mask_token_id}")
              print("-" * 30)
            lm.train()

Running on cuda
len tokenizer:  5001
There are 18507842 datapoints and 5001 token classes in the dataset
12:31:25 Training starts
12:31:30 Epoch 1, iteration 100: loss=6.1317, dev loss=5.7972
12:31:33 Epoch 1, iteration 200: loss=5.7455, dev loss=5.6817
12:31:37 Epoch 1, iteration 300: loss=5.6656, dev loss=5.6024
12:31:40 Epoch 1, iteration 400: loss=5.6024, dev loss=5.5577
12:31:43 Epoch 1, iteration 500: loss=5.5552, dev loss=5.4915
12:31:47 Epoch 1, iteration 600: loss=5.5223, dev loss=5.4866
12:31:50 Epoch 1, iteration 700: loss=5.4920, dev loss=5.4674
12:31:53 Epoch 1, iteration 800: loss=5.4996, dev loss=5.4675
12:31:56 Epoch 1, iteration 900: loss=5.4863, dev loss=5.4454
12:32:00 Epoch 1, iteration 1000: loss=5.4742, dev loss=5.4312
12:32:03 Epoch 1, iteration 1100: loss=5.4730, dev loss=5.3911
12:32:06 Epoch 1, iteration 1200: loss=5.4491, dev loss=5.4209
12:32:09 Epoch 1, iteration 1300: loss=5.4291, dev loss=5.4351
12:32:13 Epoch 1, iteration 1400: loss=5.4172, dev loss=5.37

KeyboardInterrupt: 

### Use the model to generate stories
Try varying the prompt, max tokens, temperature, and top-k sampling to study the effects.

In [ ]:
### MODIFY ###

def predict_masked(model, tokenizer, prompt="The cat sat in the chair that was by the lake."):
    model.eval()
    device = next(model.parameters()).device
    mask_token_id = tokenizer.vocab_size - 1  
    
    _, token_ids = tokenizer.tokenize(prompt)
    token_ids = torch.tensor(token_ids, dtype=torch.long, device=device)
    
    # manual amsk
    masked_input = token_ids.clone()
    positions_to_mask = [7, 9] 
    for pos in positions_to_mask:
        masked_input[pos] = mask_token_id
    
    # forward pass
    with torch.no_grad():
        logits = model(masked_input.unsqueeze(0))  
        predictions = logits.argmax(dim=-1).squeeze(0) 
    
  
    print("-" * 40)
    print(f"Prompt: {prompt}")
    print("-" * 40)
    for pos in positions_to_mask:
            original_word  = tokenizer.vocab[token_ids[pos].item()]
            predicted_word = tokenizer.vocab[predictions[pos].item()]
            is_masked = (masked_input[pos].item() == mask_token_id)
            print(f" Position {pos}: original='{original_word}' | predicted='{predicted_word}' | is [MASK]?: {is_masked}")
    print("-" * 40)

checkpoint_path = "last_checkpoint_unmasking.pt"
tokenizer_path = "/content/tokenizer.json"
tokenizer_path = "/datasets/dd2417/tokenizer.json"

tokenizer = TinyStoriesTokenizer.load(tokenizer_path)
# Add [MASK] to tokens
mask_token = "[MASK]"
if mask_token not in tokenizer.vocab:
    tokenizer.vocab.append(mask_token)
    tokenizer.vocab_size += 1
mask_id = len(tokenizer.vocab) - 1  # new token for unmasking


model = TinyStoriesLM.load(checkpoint_path, device='cuda')
print(f"Tokenizer vocab_size: {tokenizer.vocab_size}")
print(f"Model vocab_size: {model.config.vocab_size}")

predict_masked(model, tokenizer)


In [ ]:
print(mask_id)